# ⚖️ Native Tool Calling vs. Text-Based ReAct

## Learning Objectives
In this notebook, you will learn:
1. **Two ways to drive the same loop** - Native tool calls vs. a model writing `Thought / Action / Observation` text
2. **A native tool-calling agent** - `create_tool_calling_agent` + `AgentExecutor`, where the provider returns structured tool calls
3. **A text-based ReAct agent** - `create_react_agent` with the original ReAct prompt, where LangChain parses the model's text
4. **Choosing between them** - What each costs, where each breaks, and what LangChain 1.x's `create_agent` uses

## Where this fits
`01_react_agent_langgraph.ipynb` built the ReAct loop in LangGraph on top of native tool calls. The ReAct paper predates native tool calling: it drove the loop by asking the model to *write* its reasoning and actions as text, which the framework then parsed. This notebook runs both drivers on the same tools so you can see the difference in the trace. Both agents here use LangChain 0.x's `AgentExecutor`, which lives on in the `langchain_classic` package; `03_react_with_create_agent.ipynb` moves to the 1.x API.

## Prerequisites
- The ReAct loop — `01_react_agent_langgraph.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`. The tools are deterministic stubs, so no other keys are needed.

| Property | Value |
|---|---|
| Origin (Tool Use) | OpenAI function-calling API, June 2023 — conceptual ancestor: Toolformer (Schick et al., 2023) |
| Origin (ReAct) | Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models*, ICLR 2023 ([arXiv:2210.03629](https://arxiv.org/abs/2210.03629)) |

---
## 📦 Step 1: Environment Setup

Load the API key, then create the one LLM both agents share — so any difference you see comes from the driver, not the model.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


In [2]:
# ============================================================================
# LLM INITIALIZATION: One model for both agents
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


---
## 🔧 Part 1: The Tools

Function calling (also known as tool calling) is a mechanism that allows LLMs to invoke external functions or tools in a structured way. Instead of generating free-form text responses, the model can output structured JSON that represents function calls with specific parameters.

Both agents get the same two deterministic stubs, so every run is reproducible.

In [3]:
# ============================================================================
# TOOLS: A weather lookup and a calculator (deterministic stubs)
# ============================================================================
from langchain_core.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get current weather for a location (city name, e.g. 'Tokyo')."""
    weather_data = {
        "New York": "22°C, sunny",
        "London": "15°C, rainy",
        "Tokyo": "28°C, cloudy",
    }
    return weather_data.get(location, f"Weather data not available for {location}")


@tool
def calculate(expression: str) -> str:
    """Perform basic mathematical calculations. Input: a Python arithmetic expression, e.g. '15 * 8 + 20'."""
    try:
        # No builtins: the expression can do arithmetic and nothing else
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception:
        return "Invalid calculation"


tools = [get_weather, calculate]

print(f"✅ Defined {len(tools)} tools: {[t.name for t in tools]}")

✅ Defined 2 tools: ['get_weather', 'calculate']


---
## 🔗 Part 2: The Native Tool-Calling Agent

`create_tool_calling_agent` sends the tool schemas through the provider's function-calling API. The model replies with a structured `tool_calls` field; `AgentExecutor` runs the tools, appends the results to the `agent_scratchpad`, and calls the model again until it answers without a tool call.

### Key Concepts:
- **`agent_scratchpad`**: where the executor puts previous tool calls and their results for the next model call
- **`AgentExecutor`**: the 0.x loop runner — the same job the LangGraph graph did in `01_react_agent_langgraph.ipynb`
- **`return_intermediate_steps=True`**: keeps every `(action, observation)` pair so we can inspect the trace

In [4]:
# ============================================================================
# NATIVE TOOL CALLING: create_tool_calling_agent + AgentExecutor
# ============================================================================
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# The generic tool-calling prompt (the content of hub prompt "hwchase17/openai-tools-agent")
tool_calling_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

tool_calling_agent = create_tool_calling_agent(llm, tools, tool_calling_prompt)
tool_calling_executor = AgentExecutor(
    agent=tool_calling_agent, tools=tools, verbose=True, return_intermediate_steps=True
)

print("✅ Tool-calling agent ready")

✅ Tool-calling agent ready


In [5]:
# ============================================================================
# TEST: Three queries through the native tool-calling agent
# ============================================================================
queries = [
    "What's the weather in Tokyo?",
    "Calculate 15 * 8 + 20",
    "Get weather for London and calculate 25% of 80",
]

for query in queries:
    print(f"\n❓ Query: {query}")
    response = tool_calling_executor.invoke({"input": query})
    print(f"🤖 Response: {response['output']}")
    print(f"🔧 Tool calls made: {[step[0].tool for step in response['intermediate_steps']]}")


❓ Query: What's the weather in Tokyo?


> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'location': 'Tokyo'}`


28°C, cloudyThe weather in Tokyo is currently 28°C and cloudy.

> Finished chain.
🤖 Response: The weather in Tokyo is currently 28°C and cloudy.
🔧 Tool calls made: ['get_weather']

❓ Query: Calculate 15 * 8 + 20


> Entering new AgentExecutor chain...

Invoking: `calculate` with `{'expression': '15 * 8 + 20'}`


Result: 140The result of the calculation \( 15 \times 8 + 20 \) is 140.

> Finished chain.
🤖 Response: The result of the calculation \( 15 \times 8 + 20 \) is 140.
🔧 Tool calls made: ['calculate']

❓ Query: Get weather for London and calculate 25% of 80


> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'location': 'London'}`


15°C, rainy
Invoking: `calculate` with `{'expression': '0.25 * 80'}`


Result: 20.0The current weather in London is 15°C and rainy. Additionally, 25% of 80 is 20.0.

> Finished chain.
🤖 Response: The 

Look at the third query: the model asked for `get_weather` **and** `calculate` in a single turn (parallel tool calls), then answered. Nothing was parsed from text — the tool names and arguments arrived as structured data.

---
## 💭 Part 3: The Text-Based ReAct Agent

The original ReAct recipe works with any model that can follow instructions, even one with no tool-calling API. The prompt lists the tools as text and asks the model to write a strict format. LangChain stops generation at `Observation:`, parses `Action` and `Action Input` out of the text with a regular expression, runs the tool, pastes the result back after `Observation:`, and asks the model to continue.

In [6]:
# ============================================================================
# THE REACT PROMPT: The content of hub prompt "hwchase17/react"
# ============================================================================
from langchain_core.prompts import PromptTemplate

react_prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

print(f"✅ ReAct prompt variables: {react_prompt.input_variables}")

✅ ReAct prompt variables: ['agent_scratchpad', 'input', 'tool_names', 'tools']


In [7]:
# ============================================================================
# TEXT-BASED REACT: create_react_agent + AgentExecutor
# ============================================================================
from langchain_classic.agents import create_react_agent

react_agent = create_react_agent(llm, tools, react_prompt)
react_executor = AgentExecutor(
    agent=react_agent,
    tools=tools,
    verbose=True,
    return_intermediate_steps=True,
    # If the model breaks the format, send the parse error back as an observation
    handle_parsing_errors=True,
    # Stop a confused run early (the default is 15 steps)
    max_iterations=6,
)

react_query = "What's the weather in Tokyo and calculate 25% of 80?"
react_response = react_executor.invoke({"input": react_query})

print(f"\n🤖 ReAct Response: {react_response['output']}")



> Entering new AgentExecutor chain...
I need to get the current weather for Tokyo and also calculate 25% of 80. I'll start with the weather.  
Action: get_weather  
Action Input: 'Tokyo'  Weather data not available for 'Tokyo'It seems I couldn't retrieve the weather data for Tokyo. However, I can still calculate 25% of 80.  
Action: calculate  
Action Input: '0.25 * 80'  Result: 0.25 * 80The calculation for 25% of 80 is straightforward. I will perform the calculation now.  
Action: calculate  
Action Input: '0.25 * 80'  Result: 0.25 * 80It seems there was an error in retrieving the calculation result. I will try to perform the calculation again.  
Action: calculate  
Action Input: '0.25 * 80'  Result: 0.25 * 80It appears that I'm unable to retrieve the calculation result for 25% of 80. I will try to perform the calculation one more time.  
Action: calculate  
Action Input: '0.25 * 80'  Result: 0.25 * 80It seems that I'm unable to retrieve the calculation result for 25% of 80. I will 

### 3.1 🔎 What the Model Actually Wrote

Each intermediate step keeps the raw text the model produced. This is the "reasoning" half of ReAct made visible — and it is also exactly the text a regex had to parse correctly for the run to continue.

In [8]:
# ============================================================================
# INSPECT: The raw Thought / Action text behind each step
# ============================================================================
for i, (action, observation) in enumerate(react_response["intermediate_steps"], 1):
    print(f"--- Step {i} ---")
    print(f"📝 Model text:\n{action.log.strip()}")
    print(f"👀 Observation: {observation}\n")

--- Step 1 ---
📝 Model text:
I need to get the current weather for Tokyo and also calculate 25% of 80. I'll start with the weather.  
Action: get_weather  
Action Input: 'Tokyo'
👀 Observation: Weather data not available for 'Tokyo'

--- Step 2 ---
📝 Model text:
It seems I couldn't retrieve the weather data for Tokyo. However, I can still calculate 25% of 80.  
Action: calculate  
Action Input: '0.25 * 80'
👀 Observation: Result: 0.25 * 80

--- Step 3 ---
📝 Model text:
The calculation for 25% of 80 is straightforward. I will perform the calculation now.  
Action: calculate  
Action Input: '0.25 * 80'
👀 Observation: Result: 0.25 * 80

--- Step 4 ---
📝 Model text:
It seems there was an error in retrieving the calculation result. I will try to perform the calculation again.  
Action: calculate  
Action Input: '0.25 * 80'
👀 Observation: Result: 0.25 * 80

--- Step 5 ---
📝 Model text:
It appears that I'm unable to retrieve the calculation result for 25% of 80. I will try to perform the calcul

Read the `Action Input` lines and the observations next to them. With `gpt-4o-mini` this run usually goes wrong in an instructive way: the model writes `Action Input: 'Tokyo'` **with quotes**, the parser passes the text through verbatim, and the tool looks up the city `'Tokyo'` — quotes included — and finds nothing. The calculator receives `'0.25 * 80'`, a quoted *string*, and "evaluates" it to itself. The model sees odd observations, retries, and burns steps until `max_iterations` stops it.

Nothing is wrong with the model's reasoning — it knows exactly which tool it wants. The failure is in the **transport**: text is ambiguous, and the tool gets whatever characters the model happened to write. (If your run went cleanly, the model left the quotes off this time; the next section is still how text-driven tools are made robust.)

### 3.2 🛡️ Hardening the Tools for Text Input

A text-driven agent needs tools that tolerate the formatting noise a model adds: stray quotes, whitespace, trailing punctuation. Wrap the same logic with input cleaning and run the same question again.

In [9]:
# ============================================================================
# HARDENED TOOLS: Normalize text input before using it
# ============================================================================
def clean(text: str) -> str:
    """Strip whitespace and the quotes a model often wraps around Action Input."""
    return text.strip().strip("'\"").strip()


@tool("get_weather")
def get_weather_text(location: str) -> str:
    """Get current weather for a location (city name, e.g. Tokyo)."""
    return get_weather.invoke(clean(location))


@tool("calculate")
def calculate_text(expression: str) -> str:
    """Perform basic mathematical calculations. Input: a Python arithmetic expression, e.g. 15 * 8 + 20."""
    return calculate.invoke(clean(expression))


text_tools = [get_weather_text, calculate_text]
react_executor = AgentExecutor(
    agent=create_react_agent(llm, text_tools, react_prompt),
    tools=text_tools,
    verbose=True,
    return_intermediate_steps=True,
    handle_parsing_errors=True,
    max_iterations=6,
)

react_response = react_executor.invoke({"input": react_query})
print(f"\n🤖 ReAct Response: {react_response['output']}")
print(f"🔧 Steps taken: {len(react_response['intermediate_steps'])}")



> Entering new AgentExecutor chain...
I need to get the current weather in Tokyo and also calculate 25% of 80. I'll start by getting the weather information.

Action: get_weather  
Action Input: "Tokyo"  28°C, cloudyI have the current weather in Tokyo, which is 28°C and cloudy. Now, I need to calculate 25% of 80.

Action: calculate  
Action Input: "0.25 * 80"  Result: 20.0I now know the final answer, which includes both the weather in Tokyo and the calculation result. 

Final Answer: The weather in Tokyo is 28°C and cloudy, and 25% of 80 is 20.

> Finished chain.

🤖 ReAct Response: The weather in Tokyo is 28°C and cloudy, and 25% of 80 is 20.
🔧 Steps taken: 2


Same model, same prompt, same reasoning — and now it finishes, because the tools no longer depend on the model's punctuation. Compare the step count with Part 2: there, one model call returned **two** structured tool calls. Here, the format allows one `Action` per turn, so the same question takes one model call per tool plus a final one — and every step depends on the model reproducing the format exactly.

---
## 🧭 Part 4: Choosing a Driver

| | Native tool calling | Text-based ReAct |
|---|---|---|
| How the tool call travels | Structured `tool_calls` from the provider API | Free text, parsed with a regex |
| Needs a tool-calling model | Yes | No — any instruction-following model |
| Parallel calls in one turn | Yes | No, one `Action` per turn |
| Visible reasoning | Optional (the model may emit none) | Always — the `Thought:` lines |
| Typical failure | Wrong tool or bad arguments | Format drift → `OutputParserException`, or arguments polluted by quotes/whitespace |

Native tool calling won: it is what `create_agent` in LangChain 1.x uses, and there is no text to parse and nothing to fail parsing. The ReAct *idea* — interleave reasoning with acting, and read every observation before the next step — is unchanged; only the transport changed. The cell below runs the same question through `create_agent`, the 1.x replacement for both executors above.

In [10]:
# ============================================================================
# THE 1.x EQUIVALENT: create_agent runs the native tool-calling loop
# ============================================================================
from langchain.agents import create_agent

modern_agent = create_agent(llm, tools, system_prompt="You are a helpful assistant")
result = modern_agent.invoke({"messages": [("user", react_query)]})

model_calls = sum(1 for m in result["messages"] if m.type == "ai")
print(f"🤖 Answer: {result['messages'][-1].content}")
print(f"📞 Model calls: {model_calls}   (the text-based ReAct agent needed "
      f"{len(react_response['intermediate_steps']) + 1})")

🤖 Answer: The current weather in Tokyo is 28°C and cloudy. Additionally, 25% of 80 is 20.0.
📞 Model calls: 2   (the text-based ReAct agent needed 3)


---
## 📝 Summary

In this notebook, we learned:

### 1. Two Ways to Drive the Same Loop
- Both agents think, act, observe and repeat; they differ only in how a tool call gets from the model to your code

### 2. The Native Tool-Calling Agent
- The provider returns structured `tool_calls`; several can arrive in one turn
- `AgentExecutor` (now in `langchain_classic`) runs the loop and keeps `intermediate_steps`

### 3. The Text-Based ReAct Agent
- The prompt lists tools as text and demands a `Thought / Action / Action Input / Observation` format
- Works with any model, shows its reasoning, but pays with extra model calls and parse failures (`handle_parsing_errors=True` turns them into observations)
- Tools driven by text must clean their input: a quoted `Action Input` is passed through verbatim

### 4. Choosing a Driver
- Use native tool calling whenever the model supports it; `create_agent` in LangChain 1.x does
- Text-based ReAct remains useful for models without a tool-calling API, and for seeing what "reasoning" means in the pattern

### Next Steps
- **`03_react_with_create_agent.ipynb`**: the ReAct loop prepackaged by LangChain 1.x — tools, call limits and memory